# Notebook 2 — Data Preprocessing: TIF → PNG + Batch Splitting
**MAI 656 — Natural Language Processing | Canadian University Dubai | Spring 2026**

This notebook prepares raw TIF images for the GPT labeling pipeline:

1. **Convert** all `.tif` / `.tiff` files in `data/raw/` to PNG → `data/raw_png/`
2. **Split** the PNGs into fixed-size batch folders → `data/batches/batch_001/`, `batch_002/`, …

> 💡 **No GPU required.** Run this locally or on free Colab.

**Input:** `data/raw/` — folder of `.tif` / `.tiff` images  
**Output:**
- `data/raw_png/` — all converted PNGs (flat)
- `data/batches/batch_NNN/` — PNGs split into sub-folders of `BATCH_SIZE` each

## 1. Install Dependencies

In [1]:
!pip install Pillow tqdm --quiet

## 2. Set Project Root

**Colab:** mounts Google Drive and reads from `MyDrive/nlp_project`.  
**Local:** update `LOCAL_PROJECT_ROOT` below to point to your project folder.

In [ ]:
import os
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/nlp_project')
else:
    # ── Set this to your local project directory ──────────────────────────
    PROJECT_ROOT = Path(r'C:\Users\mitah\github_projects\nlp_project')
    # ──────────────────────────────────────────────────────────────────────

assert PROJECT_ROOT.exists(), (
    f'Project root not found: {PROJECT_ROOT}\n'
    'Update LOCAL_PROJECT_ROOT above to match your system.'
)

print(f'{"Colab" if IN_COLAB else "Local"} | Project root: {PROJECT_ROOT}')

## 3. Configuration

Adjust `BATCH_SIZE` to control how many images land in each batch folder.  
Smaller batches = easier to resume if a GPT labeling run is interrupted.

In [ ]:
from pathlib import Path

RAW_DIR    = Path(PROJECT_ROOT) / 'data' / 'raw'       # TIF source
PNG_DIR    = Path(PROJECT_ROOT) / 'data' / 'raw_png'   # flat PNG output
BATCH_DIR  = Path(PROJECT_ROOT) / 'data' / 'batches'   # batched PNG folders

BATCH_SIZE = 100   # images per batch folder

PNG_DIR.mkdir(parents=True, exist_ok=True)
BATCH_DIR.mkdir(parents=True, exist_ok=True)

print(f'Source TIFs:  {RAW_DIR}')
print(f'PNG output:   {PNG_DIR}')
print(f'Batch output: {BATCH_DIR}')
print(f'Batch size:   {BATCH_SIZE}')

## 4. Step 1 — Convert TIF → PNG (Parallel)

Converts every `.tif` / `.tiff` file in `data/raw/` to a PNG in `data/raw_png/` using a thread pool.  
Already-converted files are skipped upfront — safe to re-run.  
Adjust `MAX_WORKERS` if you want to throttle CPU/disk usage.

In [ ]:
import os
import time
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# Parallel workers — scales to all logical CPUs by default.
# Lower this (e.g. MAX_WORKERS = 4) if disk I/O becomes the bottleneck.
MAX_WORKERS = os.cpu_count()

tif_files = sorted(RAW_DIR.glob('*.tif')) + sorted(RAW_DIR.glob('*.tiff'))

if not tif_files:
    print(f'No TIF/TIFF files found in {RAW_DIR}.')
    print('Upload your .tif files to data/raw/ and re-run this cell.')
else:
    # Filter out files that are already converted so workers never duplicate work
    to_convert = [f for f in tif_files
                  if not (PNG_DIR / (f.stem + '.png')).exists()]
    n_skipped = len(tif_files) - len(to_convert)

    print(f'Found       : {len(tif_files)} TIF file(s)')
    print(f'Already done: {n_skipped}')
    print(f'To convert  : {len(to_convert)}')
    print(f'Workers     : {MAX_WORKERS}')

    def convert_one(tif_path: Path) -> tuple:
        """Convert a single TIF to PNG. Returns (filename, error_or_None)."""
        out = PNG_DIR / (tif_path.stem + '.png')
        try:
            with Image.open(tif_path) as img:
                if img.mode not in ('RGB', 'L'):
                    img = img.convert('RGB')
                img.save(out, format='PNG')
            return (tif_path.name, None)
        except Exception as exc:
            return (tif_path.name, str(exc))

    converted = 0
    errors    = 0
    t_start   = time.perf_counter()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(convert_one, f): f for f in to_convert}
        with tqdm(total=len(to_convert), desc='Converting', unit='img') as pbar:
            for future in as_completed(futures):
                name, err = future.result()
                if err:
                    tqdm.write(f'  ERROR {name}: {err}')
                    errors += 1
                else:
                    converted += 1
                pbar.update(1)

    elapsed = time.perf_counter() - t_start
    per_img = elapsed / max(len(to_convert), 1) * 1000

    print(f'\nDone in {elapsed:.1f}s  ({per_img:.0f} ms/image avg)')
    print(f'  Converted: {converted}')
    print(f'  Skipped (already existed): {n_skipped}')
    print(f'  Errors: {errors}')

## 5. Verify Conversion

In [ ]:
all_pngs = sorted(PNG_DIR.glob('*.png'))
print(f'Total PNGs in {PNG_DIR}: {len(all_pngs)}')

if len(tif_files) > 0 and len(all_pngs) != len(tif_files):
    print(f'WARNING: {len(tif_files)} TIFs but only {len(all_pngs)} PNGs — check errors above.')

# Preview first 5 filenames
for p in all_pngs[:5]:
    print(f'  {p.name}')
if len(all_pngs) > 5:
    print(f'  ... and {len(all_pngs) - 5} more')

## 6. (Optional) Preview a Sample Image

In [ ]:
if all_pngs:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, min(4, len(all_pngs)), figsize=(16, 4))
    if len(all_pngs) == 1:
        axes = [axes]

    for ax, img_path in zip(axes, all_pngs[:4]):
        img = Image.open(img_path)
        ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
        ax.set_title(img_path.name, fontsize=8)
        ax.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print('No PNGs to preview.')

## 7. Step 2 — Split PNGs into Batch Folders

Creates `data/batches/batch_001/`, `batch_002/`, … each containing at most `BATCH_SIZE` PNGs.  
Files are **copied** (not moved) so `data/raw_png/` stays intact.  
Already-batched files are skipped — safe to re-run.

In [ ]:
import shutil
import math
from tqdm.auto import tqdm

if not all_pngs:
    print('No PNGs found. Run the conversion step first.')
else:
    num_batches = math.ceil(len(all_pngs) / BATCH_SIZE)
    print(f'Splitting {len(all_pngs)} PNGs into {num_batches} batches of up to {BATCH_SIZE} each...')

    copied = 0
    skipped = 0

    for batch_idx in tqdm(range(num_batches), desc='Batching'):
        batch_folder = BATCH_DIR / f'batch_{batch_idx + 1:03d}'
        batch_folder.mkdir(exist_ok=True)

        start = batch_idx * BATCH_SIZE
        end   = start + BATCH_SIZE
        batch_files = all_pngs[start:end]

        for src in batch_files:
            dst = batch_folder / src.name
            if dst.exists():
                skipped += 1
            else:
                shutil.copy2(src, dst)
                copied += 1

    print(f'\nDone.')
    print(f'  Copied:  {copied}')
    print(f'  Skipped (already exist): {skipped}')

## 8. Verify Batch Structure

In [ ]:
batch_folders = sorted(BATCH_DIR.glob('batch_*'))

print(f'Total batch folders: {len(batch_folders)}')
print(f'{"Batch":<12} {"Images":>8}')
print('-' * 22)

total_batched = 0
for folder in batch_folders:
    count = len(list(folder.glob('*.png')))
    total_batched += count
    print(f'{folder.name:<12} {count:>8}')

print('-' * 22)
print(f'{"TOTAL":<12} {total_batched:>8}')

if total_batched != len(all_pngs):
    print(f'\nWARNING: {len(all_pngs)} PNGs but {total_batched} batched — mismatch!')

## Summary

| Path | Contents |
|------|----------|
| `data/raw/` | Original TIF files (untouched) |
| `data/raw_png/` | All converted PNGs (flat) |
| `data/batches/batch_NNN/` | PNGs split into batches of `BATCH_SIZE` |

## Next Step

**Notebook 3 → `03_data_collection.ipynb`** — Label each batch with GPT-4o to generate transcriptions